# Agent 1 — Teams Alert Dispatcher

Reads completed validation runs that have not yet been alerted,
builds a formatted Teams Adaptive Card for each run,
posts the card to the configured Teams channel,
and marks each run as alerted to prevent duplicates.

**Runs after 07_ai_summarization completes.**
**Safe to re-run — already-alerted runs are skipped.**

Reads from:
  uc1_validation_run_b
  uc1_validation_results_b

Updates:
  uc1_validation_run_b (alert_sent, alert_sent_timestamp, alert_channel)

In [0]:
import requests
import json
from datetime import datetime
from pyspark.sql import functions as F
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ── Catalog and Schema ─────────────────────────────────────────────────────

dbutils.widgets.text("catalog", "learning_dev", "Target Catalog")
dbutils.widgets.text("schema", "s819534", "Target Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")

# ── Table References ───────────────────────────────────────────────────────

VALIDATION_RUN     = f"{CATALOG}.{SCHEMA}.uc1_validation_run_b"
VALIDATION_RESULTS = f"{CATALOG}.{SCHEMA}.uc1_validation_results_b"

# ── Teams Configuration ────────────────────────────────────────────────────

# Retrieve webhook URL from Databricks Secrets
TEAMS_WEBHOOK_URL = dbutils.secrets.get(
    scope="agent1-secrets",
    key  ="teams-webhook-url"
)

ALERT_CHANNEL = "platform-support"

# ── Alert Routing ──────────────────────────────────────────────────────────

# Statuses that trigger an alert
ALERT_ON_STATUSES = {"CRITICAL", "WARNING"}

# ── Severity Emoji Map ─────────────────────────────────────────────────────

STATUS_EMOJI = {
    "CRITICAL": "🔴",
    "WARNING" : "🟡",
    "PASS"    : "🟢",
    "INFO"    : "🔵",
    "ERROR"   : "⚫"
}

STATUS_COLOR = {
    "CRITICAL": "attention",    # red in Teams Adaptive Cards
    "WARNING" : "warning",      # yellow
    "PASS"    : "good",         # green
    "ERROR"   : "dark"          # dark gray
}

print("=" * 60)
print("AGENT 1 — TEAMS ALERT DISPATCHER")
print("=" * 60)
print(f"Catalog        : {CATALOG}")
print(f"Schema         : {SCHEMA}")
print(f"Alert channel  : {ALERT_CHANNEL}")
print(f"Alert statuses : {ALERT_ON_STATUSES}")
print(f"Webhook loaded : {'YES' if TEAMS_WEBHOOK_URL else 'NO'}")
print(f"Started at     : {datetime.now()}")
print("=" * 60)

## Step 1: Load Runs That Need Alerts

In [0]:
from pyspark.sql import functions as F

def load_runs_to_alert() -> list:
    """
    Loads validation runs that meet all three conditions:
    1. Overall status is CRITICAL or WARNING
    2. AI enrichment is COMPLETED or NOT_REQUIRED
    3. Alert has not already been sent

    Ordered by business_date and table name for
    consistent processing order.
    """
    df = (
        spark.table(VALIDATION_RUN)
             .filter(
                 F.col("overall_status").isin(list(ALERT_ON_STATUSES)) &
                 F.col("ai_enrichment_status").isin(
                     ["COMPLETED", "NOT_REQUIRED"]
                 ) &
                 (
                     F.col("alert_sent").isNull() |
                     (F.col("alert_sent") == False)
                 )
             )
             .orderBy("business_date", "fully_qualified_name")
    )

    runs = [row.asDict() for row in df.collect()]

    print(f"Runs requiring alerts: {len(runs)}")

    for r in runs:
        emoji = STATUS_EMOJI.get(r["overall_status"], "⚪")
        print(
            f"  {emoji} {r['business_date']}  "
            f"{r['fully_qualified_name'].split('.')[-1]:<35}  "
            f"Status: {r['overall_status']:<10}  "
            f"AI: {r['ai_enrichment_status']}"
        )

    return runs


runs_to_alert = load_runs_to_alert()

# ── Graceful handling when no runs need alerting ───────────────────────────
# DO NOT use dbutils.notebook.exit() here.
# In a Databricks Workflow context, notebook.exit() with a message
# can interfere with downstream task dependency evaluation.
# We set a flag and handle it gracefully in the dispatch loop.

if not runs_to_alert:
    print()
    print("No runs require alerts at this time.")
    print("This is expected when:")
    print("  - All runs passed (no alert threshold crossed)")
    print("  - All failing runs were already alerted in a previous run")
    print("  - AI enrichment is still PENDING for some runs")
    print()
    print("Notebook will complete gracefully without sending alerts.")
    NO_RUNS_TO_ALERT = True
else:
    NO_RUNS_TO_ALERT = False
    print(f"\nProceeding to dispatch {len(runs_to_alert)} alert(s).")

## Step 2: Load Check Results for Each Run

In [0]:
def load_check_results(validation_run_id: str) -> list:
    """
    Loads all check results for a validation run.
    Returns all checks ordered by severity then check type.
    """
    df = (
        spark.table(VALIDATION_RESULTS)
             .filter(
                 F.col("validation_run_id") == validation_run_id
             )
             .orderBy(
                 F.when(F.col("check_status") == "CRITICAL", 1)
                  .when(F.col("check_status") == "WARNING",  2)
                  .when(F.col("check_status") == "ERROR",    3)
                  .otherwise(4),
                 "check_type"
             )
    )
    return [row.asDict() for row in df.collect()]


def load_failed_checks(validation_run_id: str) -> list:
    """
    Loads only the failed checks (WARNING and CRITICAL)
    for inclusion in the alert body.
    """
    df = (
        spark.table(VALIDATION_RESULTS)
             .filter(
                 (F.col("validation_run_id") == validation_run_id) &
                 (F.col("check_status").isin(["WARNING", "CRITICAL", "ERROR"]))
             )
             .orderBy(
                 F.when(F.col("check_status") == "CRITICAL", 1)
                  .when(F.col("check_status") == "WARNING",  2)
                  .otherwise(3),
                 "check_type"
             )
    )
    return [row.asDict() for row in df.collect()]


print("Check result loaders ready.")

## Step 3: Build Teams MessageCard Payload

In [0]:
def get_theme_color(status: str) -> str:
    """
    Returns hex color code for the card's left border.
    """
    colors = {
        "CRITICAL": "FF0000",   # Red
        "WARNING" : "FFA500",   # Orange
        "PASS"    : "00FF00",   # Green
        "ERROR"   : "808080"    # Gray
    }
    return colors.get(status, "0078D4")   # Default Microsoft blue


def format_check_detail(check: dict) -> str:
    """
    Formats a single check result as a readable string
    for the MessageCard body.
    """
    emoji      = STATUS_EMOJI.get(check["check_status"], "⚪")
    check_type = check["check_type"]
    col        = check["column_name"] or "table-level"
    observed   = check["observed_value"]
    deviation  = check["deviation_percentage"]
    warn       = check["threshold_warning"]
    crit       = check["threshold_critical"]

    if check_type == "VOLUME_ANOMALY":
        obs_str = (
            f"{int(observed):,} rows "
            f"({deviation:.1f}% from baseline)"
        )
    else:
        obs_str = (
            f"{observed:.2f}% "
            f"(deviation {deviation:.2f}%)"
        )

    return (
        f"{emoji} {check_type} on {col}: "
        f"observed {obs_str} | "
        f"warn ≥ {warn} | crit ≥ {crit}"
    )


def build_message_card(run_record: dict, failed_checks: list) -> dict:
    """
    Builds a Microsoft Teams MessageCard payload.

    MessageCard is the legacy format but is universally
    supported across all Teams versions and channel types.
    Unlike Adaptive Cards it does not require a specific
    client version.

    Reference:
    https://learn.microsoft.com/en-us/outlook/actionable-messages/message-card-reference
    """
    fqn          = run_record["fully_qualified_name"]
    table_name   = fqn.split(".")[-1]
    biz_date     = str(run_record["business_date"])
    status       = run_record["overall_status"]
    run_id       = run_record["validation_run_id"]
    total_run    = run_record["total_checks_run"]
    total_passed = run_record["total_checks_passed"]
    total_warned = run_record["total_checks_warned"]
    total_crit   = run_record["total_checks_critical"]
    ai_status    = run_record["ai_enrichment_status"]

    emoji       = STATUS_EMOJI.get(status, "⚪")
    theme_color = get_theme_color(status)
    detected_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S UTC")

    # Get AI enrichment from first failed check that has it
    ai_summary    = None
    ai_action     = None
    ai_category   = None
    ai_confidence = None

    for check in failed_checks:
        if check.get("ai_summary"):
            ai_summary    = check["ai_summary"]
            ai_action     = check["ai_recommended_action"]
            ai_category   = check["ai_failure_category"]
            ai_confidence = check["ai_confidence_score"]
            break

    # ── Build sections ─────────────────────────────────────────────────────

    sections = []

    # ── Section 1: Header and table info ──────────────────────────────────
    sections.append({
        "activityTitle"   : (
            f"{emoji} {status} — Data Quality Alert"
        ),
        "activitySubtitle": (
            "Agent 1 — Raw Ingestion Validation"
        ),
        "activityImage"   : (
            "https://adaptivecards.io/content/cats/1.png"
        ),
        "facts"           : [
            {
                "name" : "Table",
                "value": table_name
            },
            {
                "name" : "Full Name",
                "value": fqn
            },
            {
                "name" : "Business Date",
                "value": biz_date
            },
            {
                "name" : "Detected At",
                "value": detected_at
            },
            {
                "name" : "Overall Status",
                "value": f"{emoji} {status}"
            }
        ],
        "markdown": True
    })

    # ── Section 2: Check summary ───────────────────────────────────────────
    sections.append({
        "title": "📊 Check Summary",
        "facts": [
            {
                "name" : "Total Checks Run",
                "value": str(total_run)
            },
            {
                "name" : "✅ Passed",
                "value": str(total_passed)
            },
            {
                "name" : "🟡 Warning",
                "value": str(total_warned)
            },
            {
                "name" : "🔴 Critical",
                "value": str(total_crit)
            }
        ],
        "markdown": True
    })

    # ── Section 3: Failed checks ───────────────────────────────────────────
    if failed_checks:
        failed_lines = "\n\n".join([
            format_check_detail(c) for c in failed_checks
        ])

        sections.append({
            "title": f"❌ Checks Failed ({len(failed_checks)})",
            "text" : failed_lines,
            "markdown": True
        })

    # ── Section 4: AI summary (if available) ──────────────────────────────
    if ai_summary:
        sections.append({
            "title": "🤖 AI Summary",
            "text" : ai_summary,
            "markdown": True
        })

        if ai_action:
            sections.append({
                "title": "✅ Recommended Action",
                "text" : ai_action,
                "markdown": True
            })

        sections.append({
            "title": "AI Enrichment Details",
            "facts": [
                {
                    "name" : "Failure Category",
                    "value": ai_category or "N/A"
                },
                {
                    "name" : "AI Confidence",
                    "value": (
                        f"{ai_confidence:.0%}"
                        if ai_confidence else "N/A"
                    )
                },
                {
                    "name" : "AI Status",
                    "value": ai_status
                }
            ],
            "markdown": True
        })

    # ── Section 5: Footer ──────────────────────────────────────────────────
    sections.append({
        "title": "🔍 Audit",
        "facts": [
            {
                "name" : "Validation Run ID",
                "value": run_id
            },
            {
                "name" : "Catalog.Schema",
                "value": f"{CATALOG}.{SCHEMA}"
            }
        ],
        "markdown": True
    })

    # ── Assemble full MessageCard payload ──────────────────────────────────
    card = {
        "@type"     : "MessageCard",
        "@context"  : "http://schema.org/extensions",
        "themeColor": theme_color,
        "summary"   : (
            f"Agent 1 Alert: {table_name} {status} "
            f"on {biz_date}"
        ),
        "title"     : (
            f"{emoji} {status}: {table_name} — {biz_date}"
        ),
        "sections"  : sections
    }

    return card


print("MessageCard builder ready.")

## Step 4: Send Alert to Teams

In [0]:
def send_teams_alert(
    card_payload      : dict,
    validation_run_id : str
) -> bool:
    """
    Posts the Adaptive Card to the Teams webhook.
    Returns True on success, False on failure.
    """
    try:
        response = requests.post(
            TEAMS_WEBHOOK_URL,
            headers={"Content-Type": "application/json"},
            data=json.dumps(card_payload),
            timeout=30
        )

        if response.status_code in (200, 202):
            logger.info(
                f"Alert sent successfully for run {validation_run_id}. "
                f"Status: {response.status_code}"
            )
            return True
        else:
            logger.error(
                f"Alert failed for run {validation_run_id}. "
                f"Status: {response.status_code}. "
                f"Response: {response.text[:300]}"
            )
            return False

    except requests.exceptions.Timeout:
        logger.error(
            f"Alert timed out for run {validation_run_id}"
        )
        return False
    except Exception as e:
        logger.error(
            f"Unexpected error sending alert for run "
            f"{validation_run_id}: {str(e)}"
        )
        return False


def mark_alert_sent(
    validation_run_id : str,
    success           : bool
) -> None:
    """
    Updates the validation run record to mark
    the alert as sent (or failed).
    Prevents duplicate alerts on re-runs.
    """
    alert_sent_val    = "true"  if success else "false"
    alert_channel_val = ALERT_CHANNEL if success else "FAILED"

    spark.sql(f"""
        UPDATE {VALIDATION_RUN}
        SET
            alert_sent              = {alert_sent_val},
            alert_sent_timestamp    = '{datetime.now()}',
            alert_channel           = '{alert_channel_val}'
        WHERE validation_run_id     = '{validation_run_id}'
    """)


print("Alert sender ready.")

## Step 5: Dispatch Alerts for All Pending Runs

In [0]:
alerts_sent    = 0
alerts_failed  = 0
alerts_skipped = 0

# ── Check flag set in Cell 3 ───────────────────────────────────────────────

if NO_RUNS_TO_ALERT:
    print("Skipping dispatch — no runs require alerts.")
    print("Teams alert dispatcher complete with nothing to send.")

else:
    print(f"Dispatching alerts for {len(runs_to_alert)} run(s)...\n")

    for run_record in runs_to_alert:

        run_id     = run_record["validation_run_id"]
        fqn        = run_record["fully_qualified_name"]
        table_name = fqn.split(".")[-1]
        biz_date   = run_record["business_date"]
        status     = run_record["overall_status"]
        emoji      = STATUS_EMOJI.get(status, "⚪")

        print(f"{'='*60}")
        print(f"{emoji} {table_name}  |  {biz_date}  |  {status}")
        print(f"{'='*60}")

        # ── Load check results ─────────────────────────────────────────────
        failed_checks = load_failed_checks(run_id)
        all_checks    = load_check_results(run_id)

        print(f"Failed checks : {len(failed_checks)}")
        print(f"Total checks  : {len(all_checks)}")

        # ── Build MessageCard ──────────────────────────────────────────────
        try:
            card_payload = build_message_card(run_record, failed_checks)
            print(f"Card built successfully.")
        except Exception as e:
            print(f"Card build failed: {str(e)}")
            mark_alert_sent(run_id, success=False)
            alerts_failed += 1
            print()
            continue

        # ── Send to Teams ──────────────────────────────────────────────────
        success = send_teams_alert(card_payload, run_id)

        # ── Update run record ──────────────────────────────────────────────
        mark_alert_sent(run_id, success=success)

        if success:
            alerts_sent += 1
            print(f"Alert sent ✅")
        else:
            alerts_failed += 1
            print(f"Alert failed ❌")

        print()

    print(f"Dispatch complete.")
    print(f"  Sent    : {alerts_sent}")
    print(f"  Failed  : {alerts_failed}")
    print(f"  Skipped : {alerts_skipped}")

## Step 6: Verify Alert Status

In [0]:
print("=" * 60)
print("ALERT DISPATCH VERIFICATION")
print("=" * 60)

spark.sql(f"""
    SELECT
        business_date,
        SPLIT(fully_qualified_name, '\\.')[2]  AS table_name,
        overall_status,
        ai_enrichment_status,
        alert_sent,
        alert_sent_timestamp,
        alert_channel
    FROM {VALIDATION_RUN}
    ORDER BY business_date, fully_qualified_name
""").show(20, truncate=False)

## Final Summary

In [0]:
print("\n" + "=" * 60)
print("TEAMS ALERT DISPATCHER COMPLETE")
print("=" * 60)

if NO_RUNS_TO_ALERT:
    print("Alerts sent      : 0")
    print("Alerts failed    : 0")
    print("Alerts skipped   : 0")
    print("Reason           : No runs required alerting")
else:
    print(f"Alerts sent      : {alerts_sent}")
    print(f"Alerts failed    : {alerts_failed}")
    print(f"Alerts skipped   : {alerts_skipped}")
    print(f"Channel          : {ALERT_CHANNEL}")

    if alerts_failed > 0:
        print()
        print(
            f"WARNING: {alerts_failed} alert(s) failed to send. "
            f"Re-run this notebook to retry. "
            f"Failed runs have alert_sent = false in the run table."
        )

    if alerts_sent > 0:
        print()
        print(
            f"SUCCESS: {alerts_sent} alert(s) sent to #{ALERT_CHANNEL}."
        )

print(f"Completed at     : {datetime.now()}")
print("=" * 60)